# Bird Observatory — Train Your Yard Model

This notebook trains a custom bird classifier for **YOUR** feeder camera.

**What you need:** A `training_data.zip` from your Bird Observatory  
**What you'll get:** A trained model (`yard_model_edgetpu.tflite`) for your Coral USB  
**How long:** About 5–10 minutes with GPU

---

Before you start:
1. Make sure the runtime is set to GPU: **Runtime → Change runtime type → T4 GPU**
2. Export a `training_data.zip` from your Bird Observatory dashboard
3. Run each cell in order (Shift+Enter), or click **Runtime → Run All**

> **Note:** This notebook uses transfer learning — it starts from a model already trained on millions of images, then specializes it for the birds at YOUR feeder. You don’t need to be a machine learning expert. Just follow the steps.

In [ ]:
# =============================================================================
# CELL 2: Setup
# =============================================================================
# Install the tools we need.
#
# - tensorflow: the main machine learning library
# - tensorflow_hub: lets us download a pretrained model to start from
# - tensorflow-model-optimization: helps shrink the model for the Coral USB stick
#
# The edgetpu_compiler is a separate tool (from Google) that converts the
# shrunk model into the special format the Coral USB stick understands.
# We have to install it manually because it's not in the normal Python package
# repository.
# =============================================================================

print("Installing Python packages...")
!pip install -q tensorflow==2.13.0 tensorflow_hub tensorflow-model-optimization

print("Installing Edge TPU compiler (needed to convert model for Coral USB)...")
!curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg | gpg --dearmor -o /usr/share/keyrings/coral-edgetpu-archive-keyring.gpg
!echo 'deb [signed-by=/usr/share/keyrings/coral-edgetpu-archive-keyring.gpg] https://packages.cloud.google.com/apt coral-edgetpu-stable main' | tee /etc/apt/sources.list.d/coral-edgetpu.list
!apt-get update -q && apt-get install -y -q edgetpu-compiler

import tensorflow as tf
import tensorflow_hub as hub
print(f"\nTensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("\nSetup complete!")

In [ ]:
# =============================================================================
# CELL 3: Upload & Unpack Training Data
# =============================================================================
# This cell opens a file picker so you can upload your training_data.zip.
# After uploading, it:
#   1. Unzips the file
#   2. Reads manifest.json (a summary file created by the Bird Observatory)
#   3. Prints how many images you have per species
#
# The zip has this structure:
#   train/<Species Name>/<image>.jpg  ← used to teach the model
#   test/<Species Name>/<image>.jpg   ← used to measure accuracy
#   ood/<Species Name>/<image>.jpg    ← rare species (too few to train on)
#   manifest.json                     ← summary of what’s in the zip
# =============================================================================

import json
import os
import zipfile
from pathlib import Path
from google.colab import files

print("A file picker will appear below. Select your training_data.zip file.")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"\nUploaded: {zip_name}")

print("Unzipping...")
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/training_data')

# Load the manifest
manifest_path = '/content/training_data/manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

print(f"\nExport date: {manifest.get('exported_at', 'unknown')}")
print(f"Min images per species: {manifest.get('min_images', 'unknown')}")

print("\nSpecies in training set:")
species_counts = manifest.get('species_counts', {})
for species, counts in sorted(species_counts.items()):
    train_n = counts.get('train', 0)
    test_n = counts.get('test', 0)
    print(f"  {species:<35} train={train_n:>3}  test={test_n:>3}")

ood_species = manifest.get('ood_species', [])
if ood_species:
    print(f"\nRare species (OOD test only, not trained): {', '.join(ood_species)}")

print(f"\nTotal species to classify: {len(species_counts)}")

In [ ]:
# =============================================================================
# CELL 4: Inspect Data — Visual Verification
# =============================================================================
# IMPORTANT: Before training, you should LOOK at your training images.
# Bad images (wrong species, blurry, corrupted) will quietly make your
# model worse. This cell shows you a sample grid so you can spot problems.
#
# What to check:
#   ✓ Each row’s images should all be the same species
#   ✓ Images should be close-up bird crops (not the full frame)
#   ✓ Species label should match what’s in the image
#   ✗ If you see wrong species mixed in, go back to your Observatory
#     and correct those reviews before re-exporting
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
from pathlib import Path

TRAIN_DIR = Path('/content/training_data/train')
TEST_DIR  = Path('/content/training_data/test')

species_dirs = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
CLASS_NAMES = [d.name for d in species_dirs]
NUM_CLASSES = len(CLASS_NAMES)

print(f"Found {NUM_CLASSES} species: {', '.join(CLASS_NAMES)}")
print()

# Print per-species counts
print(f"{'Species':<35} {'Train':>6} {'Test':>6}")
print("-" * 50)
for species in CLASS_NAMES:
    train_count = len(list((TRAIN_DIR / species).glob('*.jpg')))
    test_dir = TEST_DIR / species
    test_count = len(list(test_dir.glob('*.jpg'))) if test_dir.exists() else 0
    print(f"{species:<35} {train_count:>6} {test_count:>6}")

# Show a grid of sample images
SAMPLES_PER_SPECIES = 5
fig, axes = plt.subplots(
    NUM_CLASSES, SAMPLES_PER_SPECIES,
    figsize=(SAMPLES_PER_SPECIES * 2.2, NUM_CLASSES * 2.2)
)
# Make axes always 2D even with 1 species
if NUM_CLASSES == 1:
    axes = [axes]

for row_idx, species in enumerate(CLASS_NAMES):
    images = list((TRAIN_DIR / species).glob('*.jpg'))
    samples = random.sample(images, min(SAMPLES_PER_SPECIES, len(images)))
    for col_idx in range(SAMPLES_PER_SPECIES):
        ax = axes[row_idx][col_idx]
        ax.axis('off')
        if col_idx < len(samples):
            img = mpimg.imread(str(samples[col_idx]))
            ax.imshow(img)
        if col_idx == 0:
            ax.set_title(species, fontsize=9, loc='left', pad=2)

plt.suptitle('Training Data Sample — Verify each row shows the correct species',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print()
print("=" * 60)
print("VISUAL CHECK REQUIRED before proceeding")
print("=" * 60)
print("Look at the images above. Each row should show ONE species.")
print("If you see wrong birds, go fix your reviews and re-export.")
print()
print("If everything looks correct, continue to Cell 5.")

In [ ]:
# =============================================================================
# CELL 5: Build the Model
# =============================================================================
# Here’s where the machine learning magic happens — but it’s simpler than
# it sounds. We use “transfer learning”:
#
#   1. Start with MobileNetV2 — a model Google already trained on 1.2 million
#      images from across the internet. It already “knows” what feathers,
#      beaks, and wings look like at a general level.
#
#   2. Freeze it — keep all that knowledge locked in place.
#
#   3. Add our classification head — a small extra layer that learns
#      “given these features, is this a House Finch or a Chickadee?”
#
# We also set up “data augmentation” — random tweaks applied to each training
# image (slight rotation, brightness changes, horizontal flip) to help the
# model generalize. More variety = less overfitting.
# =============================================================================

import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from pathlib import Path

IMAGE_SIZE = 224
BATCH_SIZE = 32
THRESHOLD = 0.45  # Confidence threshold for OOD test

TRAIN_DIR = '/content/training_data/train'
TEST_DIR  = '/content/training_data/test'

# ---- Data generators with augmentation ----
# Augmentation is applied ONLY to training images, not test images.
# - horizontal_flip: mirror the image left-right (birds look both ways)
# - brightness_range: randomly darken/brighten (handles cloudy vs sunny days)
# - rotation_range: tilt slightly (birds aren’t always perfectly upright)
# - zoom_range: random zoom out (handles birds at different distances)
# NO vertical flip — upside-down birds don’t exist in real life

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0 / 255.0,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    rotation_range=10,
    zoom_range=[0.85, 1.0],
)

# Test images get NO augmentation — just normalize pixel values to 0–1
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0 / 255.0
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

# The class names come from the directory names inside TRAIN_DIR
CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# ---- Build the model ----
# MobileNetV2 pretrained on ImageNet — small, fast, runs on Coral USB
MOBILENET_URL = 'https://tfhub.dev/google/imagenet/mobilenet_v2_100_224/feature_vector/5'

base_model = hub.KerasLayer(
    MOBILENET_URL,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    trainable=False,  # Frozen for now — we’ll unfreeze the top later
    name='mobilenet_v2'
)

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.Dropout(0.3),            # Randomly drop 30% of signals — prevents overfitting
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')
], name='yard_classifier')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f"\nModel ready. {NUM_CLASSES} output classes.")

In [ ]:
# =============================================================================
# CELL 6: Train Phase 1 — Train the Classification Head
# =============================================================================
# In Phase 1, the base MobileNetV2 model stays FROZEN.
# Only our small classification head (the last layer) learns.
#
# Think of it like this: MobileNetV2 is an expert at recognizing visual
# features. We’re teaching a brand-new student to use those expert opinions
# to tell YOUR yard’s birds apart.
#
# 10 epochs = 10 passes through the entire training dataset.
# We use a high learning rate (0.01) because we’re starting from scratch.
# =============================================================================

print("Phase 1: Training classification head (base model frozen)")
print("This should take 2-4 minutes on GPU...\n")

history_phase1 = model.fit(
    train_gen,
    epochs=10,
    validation_data=test_gen,
    verbose=1,
)

p1_val_acc = max(history_phase1.history['val_accuracy'])
print(f"\nPhase 1 complete.")
print(f"Best validation accuracy: {p1_val_acc:.1%}")

if p1_val_acc < 0.5:
    print()
    print("WARNING: Phase 1 accuracy is very low (<50%).")
    print("This might mean your training data has quality issues.")
    print("Consider re-checking Cell 4 and verifying your images.")
else:
    print("Looking good! Proceeding to Phase 2 (fine-tuning).")

In [ ]:
# =============================================================================
# CELL 7: Train Phase 2 — Fine-Tune the Top of the Base Model
# =============================================================================
# Now we UNFREEZE the top 20% of MobileNetV2’s layers.
# This lets the base model itself make small adjustments to become better
# at YOUR specific birds — not just birds in general.
#
# We use a much lower learning rate (0.0001) because the base model
# already has good weights — we want to nudge them gently, not
# overwrite them with big changes.
#
# This is the most powerful step. It’s what makes your model outperform
# a generic bird classifier for your specific feeder birds.
# =============================================================================

print("Phase 2: Fine-tuning top 20% of base model layers")

# Unfreeze the base model so we can access its layers
# hub.KerasLayer wraps the layers, so we set trainable=True on the layer
# then manually freeze the bottom 80%
base_layer = model.get_layer('mobilenet_v2')
base_layer.trainable = True

# Count layers inside the hub model
# hub.KerasLayer exposes trainable_variables; we freeze 80% of variable groups
trainable_vars = base_layer.trainable_variables
num_vars = len(trainable_vars)
freeze_until = int(num_vars * 0.80)
for var in trainable_vars[:freeze_until]:
    var._trainable = False  # noqa: SLF001 — freeze bottom 80% of variables

unfrozen_count = num_vars - freeze_until
print(f"Unfroze top {unfrozen_count}/{num_vars} variable groups in base model")

# Recompile with a much lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Training for 10 more epochs...\n")
history_phase2 = model.fit(
    train_gen,
    epochs=10,
    validation_data=test_gen,
    verbose=1,
)

p2_val_acc = max(history_phase2.history['val_accuracy'])
final_val_acc = history_phase2.history['val_accuracy'][-1]

print(f"\nPhase 2 complete.")
print(f"Best validation accuracy: {p2_val_acc:.1%}")
print(f"Final epoch accuracy:     {final_val_acc:.1%}")

In [ ]:
# =============================================================================
# CELL 8: Evaluate — In-Distribution Test
# =============================================================================
# Now we evaluate the model on the test set — images the model has NEVER
# seen during training. This tells us how accurate it really is.
#
# We look at two things:
#   1. Classification report — precision/recall per species
#      (precision = “when I say House Finch, am I right?”)
#      (recall    = “when there IS a House Finch, do I find it?”)
#
#   2. Confusion matrix — a grid showing what species get mixed up
#      (diagonal = correct; off-diagonal = mistakes)
#
# GATE: We require >= 80% overall accuracy to continue.
# If accuracy is too low, check your training data quality.
# =============================================================================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

ACCURACY_GATE = 0.80

print("Running predictions on test set...")
test_gen.reset()
y_pred_prob = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_gen.classes

# Classification report
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)

# Overall accuracy
overall_accuracy = np.mean(y_pred == y_true)
print(f"Overall accuracy: {overall_accuracy:.1%}")

# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, NUM_CLASSES * 1.5), max(6, NUM_CLASSES * 1.3)))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title('Confusion Matrix\n(rows = actual, columns = predicted)')
plt.ylabel('Actual species')
plt.xlabel('Predicted species')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Gate check
print("\n" + "=" * 60)
if overall_accuracy >= ACCURACY_GATE:
    print(f"GATE PASSED: {overall_accuracy:.1%} >= {ACCURACY_GATE:.0%}")
    print("Accuracy is good enough. Continuing to OOD evaluation.")
    in_dist_passed = True
else:
    print(f"GATE FAILED: {overall_accuracy:.1%} < {ACCURACY_GATE:.0%}")
    print("Accuracy is too low. Suggestions:")
    print("  - Check Cell 4 — are there mislabeled images in your training data?")
    print("  - Do you have at least 15 images per species in training?")
    print("  - Species that are hard to tell apart may need more training images.")
    print("  - You can still continue, but model performance may be poor.")
    in_dist_passed = False
print("=" * 60)

# Store metrics for final report
in_dist_metrics = {
    'overall_accuracy': float(overall_accuracy),
    'gate_passed': in_dist_passed,
    'phase1_best_val_accuracy': float(p1_val_acc),
    'phase2_best_val_accuracy': float(p2_val_acc),
}

In [ ]:
# =============================================================================
# CELL 9: Evaluate — Out-of-Distribution (OOD) Test
# =============================================================================
# The OOD test checks: what happens when a bird the model has NEVER seen
# visits your feeder? A good model should say “I don’t know” (low confidence)
# rather than confidently guessing wrong.
#
# We measure “abstention rate” = % of OOD images where the model’s highest
# confidence score is below the THRESHOLD (0.45).
#
# A high abstention rate (>=95%) means the model knows what it doesn’t know.
# A low abstention rate means the model is overconfident about strangers —
# dangerous for production use.
#
# This test only runs if there is an ood/ folder in the training data.
# =============================================================================

import os
import numpy as np
from pathlib import Path

THRESHOLD = 0.45
OOD_GATE = 0.95
OOD_DIR = '/content/training_data/ood'

ood_metrics = {'skipped': True}

if not os.path.isdir(OOD_DIR) or not any(Path(OOD_DIR).iterdir()):
    print("No OOD folder found (or it’s empty).")
    print("This means all your confirmed species had enough images to train on.")
    print("OOD evaluation skipped.")
else:
    print(f"OOD folder found. Running abstention test (threshold={THRESHOLD})...")

    ood_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255.0)
    ood_gen = ood_datagen.flow_from_directory(
        OOD_DIR,
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        class_mode=None,  # No labels needed — we just want predictions
        shuffle=False,
    )

    if len(ood_gen) == 0:
        print("OOD generator is empty. Skipping.")
    else:
        ood_preds = model.predict(ood_gen, verbose=1)
        max_scores = np.max(ood_preds, axis=1)  # Highest confidence for each image
        abstained = np.sum(max_scores < THRESHOLD)
        total_ood = len(max_scores)
        abstention_rate = abstained / total_ood if total_ood > 0 else 0.0

        print(f"\nOOD images tested:  {total_ood}")
        print(f"Abstained (low conf): {abstained} ({abstention_rate:.1%})")
        print(f"Overconfident (wrong): {total_ood - abstained} ({1 - abstention_rate:.1%})")

        print("\n" + "=" * 60)
        if abstention_rate >= OOD_GATE:
            print(f"OOD GATE PASSED: {abstention_rate:.1%} >= {OOD_GATE:.0%}")
            print("Model correctly ignores unfamiliar birds. Good!")
            ood_passed = True
        else:
            print(f"OOD GATE FAILED: {abstention_rate:.1%} < {OOD_GATE:.0%}")
            print("Model is being overconfident about unfamiliar birds.")
            print("Suggestion: Lower the threshold in yard_classifier.py from")
            print(f"  YARD_THRESHOLD = {THRESHOLD}  →  YARD_THRESHOLD = 0.60")
            print("This makes the model require higher confidence before committing.")
            ood_passed = False
        print("=" * 60)

        ood_metrics = {
            'skipped': False,
            'total_ood_images': int(total_ood),
            'abstained': int(abstained),
            'abstention_rate': float(abstention_rate),
            'threshold': THRESHOLD,
            'gate_passed': ood_passed,
        }

In [ ]:
# =============================================================================
# CELL 10: Quantize & Compile for Coral USB
# =============================================================================
# The Coral USB Accelerator needs the model in a special format:
#   1. TFLite — a compact format designed for edge devices
#   2. INT8 quantized — weights stored as 8-bit integers instead of 32-bit
#      floats. This is ~4x smaller and runs much faster on the Coral.
#   3. EdgeTPU compiled — Google’s compiler maps the layers onto the Coral’s
#      special chip so it runs in hardware (not just software).
#
# “Representative dataset” = a sample of your real training images.
# The quantizer uses these to figure out the range of values in each layer
# so it can compress them accurately without losing much quality.
# =============================================================================

import tensorflow as tf
import numpy as np
import os
from pathlib import Path

SAVED_MODEL_PATH = '/content/yard_model_keras'
TFLITE_FLOAT_PATH = '/content/yard_model_float.tflite'
TFLITE_QUANT_PATH = '/content/yard_model_quant.tflite'
EDGETPU_PATH      = '/content/yard_model_edgetpu.tflite'

# Step 1: Save the full Keras model
print("Step 1: Saving Keras model...")
model.save(SAVED_MODEL_PATH)
print(f"  Saved to {SAVED_MODEL_PATH}")

# Step 2: Build a representative dataset generator
# The quantizer needs to see ~100-200 real images to calibrate
print("\nStep 2: Building representative dataset for quantization...")

def get_representative_dataset():
    """Yield batches of real training images for quantization calibration."""
    train_dir = Path('/content/training_data/train')
    all_images = list(train_dir.rglob('*.jpg'))
    # Use up to 200 images (more = better calibration, but slower)
    sample = all_images[:200]
    for img_path in sample:
        img = tf.keras.preprocessing.image.load_img(
            str(img_path),
            target_size=(IMAGE_SIZE, IMAGE_SIZE)
        )
        arr = tf.keras.preprocessing.image.img_to_array(img) / 255.0
        arr = np.expand_dims(arr, axis=0).astype(np.float32)
        yield [arr]

print(f"  Using up to 200 training images for calibration")

# Step 3: Convert to INT8 quantized TFLite
print("\nStep 3: Quantizing to INT8 TFLite (this takes a minute)...")
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = get_representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.uint8
converter.inference_output_type = tf.uint8
tflite_quant = converter.convert()

with open(TFLITE_QUANT_PATH, 'wb') as f:
    f.write(tflite_quant)
quant_size = os.path.getsize(TFLITE_QUANT_PATH) / 1024 / 1024
print(f"  Quantized model saved: {quant_size:.2f} MB")

# Step 4: Compile for Edge TPU
print("\nStep 4: Compiling for Edge TPU (Coral USB)...")
!edgetpu_compiler -s -o /content {TFLITE_QUANT_PATH}

# The compiler appends _edgetpu to the filename
compiled_name = os.path.basename(TFLITE_QUANT_PATH).replace('.tflite', '_edgetpu.tflite')
compiled_path = f'/content/{compiled_name}'

if os.path.exists(compiled_path):
    edgetpu_size = os.path.getsize(compiled_path) / 1024 / 1024
    # Rename to the standard output name
    os.rename(compiled_path, EDGETPU_PATH)
    print(f"  Edge TPU model saved: {os.path.getsize(EDGETPU_PATH) / 1024 / 1024:.2f} MB")
    print("\nQuantization and compilation complete!")
else:
    print("  WARNING: edgetpu_compiler did not produce an output file.")
    print("  Check the compiler output above for errors.")
    print(f"  You may need to use {TFLITE_QUANT_PATH} without EdgeTPU optimization.")

In [ ]:
# =============================================================================
# CELL 11: Download Results
# =============================================================================
# Almost done! This cell:
#   1. Creates a labels file (one species name per line)
#      This tells yard_classifier.py what class 0, class 1, class 2, etc. are.
#
#   2. Creates a training_report.json with all your accuracy metrics
#      so you have a record of this training run.
#
#   3. Downloads all 3 files to your computer.
#
# After downloading:
#   - Copy yard_model_edgetpu.tflite to your iMac as models/yard_model.tflite
#   - Copy yard_model_labels.txt     to your iMac as models/yard_model_labels.txt
#   - The existing yard_classifier.py will pick it up automatically on next restart.
# =============================================================================

import json
import datetime
from google.colab import files

LABELS_PATH = '/content/yard_model_labels.txt'
REPORT_PATH = '/content/training_report.json'

# Step 1: Write labels file
# One species per line, in the same order as the model’s output classes
print("Writing labels file...")
with open(LABELS_PATH, 'w') as f:
    for name in CLASS_NAMES:
        f.write(name + '\n')
print(f"  {len(CLASS_NAMES)} species written to {LABELS_PATH}")

# Step 2: Write training report
print("\nWriting training report...")
report = {
    'trained_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'in_distribution': in_dist_metrics,
    'ood': ood_metrics,
    'training': {
        'phase1_epochs': 10,
        'phase1_lr': 0.01,
        'phase1_best_val_accuracy': float(p1_val_acc),
        'phase2_epochs': 10,
        'phase2_lr': 0.0001,
        'phase2_best_val_accuracy': float(p2_val_acc),
        'final_val_accuracy': float(final_val_acc),
    },
    'model': {
        'base': 'MobileNetV2 (ImageNet pretrained)',
        'image_size': IMAGE_SIZE,
        'output_format': 'EdgeTPU INT8 TFLite',
    }
}

with open(REPORT_PATH, 'w') as f:
    json.dump(report, f, indent=2)
print(f"  Report saved to {REPORT_PATH}")

# Step 3: Download all files
print("\nDownloading files to your computer...")

import os
if os.path.exists(EDGETPU_PATH):
    files.download(EDGETPU_PATH)
    print(f"  Downloaded: yard_model_edgetpu.tflite")
else:
    print(f"  SKIPPED yard_model_edgetpu.tflite (file not found — check Cell 10)")

files.download(LABELS_PATH)
print(f"  Downloaded: yard_model_labels.txt")

files.download(REPORT_PATH)
print(f"  Downloaded: training_report.json")

print()
print("=" * 60)
print("DEPLOYMENT INSTRUCTIONS")
print("=" * 60)
print()
print("On your iMac, run these commands in Terminal:")
print()
print("  # Back up the current model")
print("  cp ~/bird-classifier/models/yard_model.tflite \\")
print("     ~/bird-classifier/models/yard_model_prev.tflite")
print()
print("  cp ~/bird-classifier/models/yard_model_labels.txt \\")
print("     ~/bird-classifier/models/yard_model_prev_labels.txt")
print()
print("  # Install the new model")
print("  cp ~/Downloads/yard_model_edgetpu.tflite \\")
print("     ~/bird-classifier/models/yard_model.tflite")
print()
print("  cp ~/Downloads/yard_model_labels.txt \\")
print("     ~/bird-classifier/models/yard_model_labels.txt")
print()
print("  # Restart the bird pipeline")
print("  launchctl stop com.vives.bird-pipeline")
print("  launchctl start com.vives.bird-pipeline")
print()
print("The yard_classifier.py will load the new model automatically.")
print("Check the dashboard to confirm it’s classifying correctly.")
print("=" * 60)